<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session05-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 5 lab: real database searches, in code {.unnumbered}

Every search in this notebook runs on a server at **EMBL-EBI**, the
European Bioinformatics Institute, through a **web API**: the notebook
sends the sequences, the server runs the program, and the notebook reads
the result back. Nothing has to be installed, so the notebook runs on
Colab (in the browser or from VS Code) exactly as it is. These are the
same services that sit behind EBI's web pages, and the same pattern
(submit a job, wait, fetch the result) works for most of the large
bioinformatics services.

Three real, runnable pieces, each producing output you will report on in
the Session 5 lab quiz on Canvas:

1. **Clustal Omega and a PSSM** on five globin-family proteins used
   throughout this course.
2. **An iterated profile search**, comparing its first round with its
   third, against every sequence in the PDB.
3. **A live Pfam domain search** for the HIV-1 Gag-Pol polyprotein,
   through EBI's InterPro API.

Run every cell from top to bottom. Nothing here is a fill-in-the-blank
exercise: run it, read the real output, and use it to answer the quiz.


## 1. Setup

Biopython (to read the alignment) and two small helper functions for
EBI's **Job Dispatcher**, which runs tools such as Clustal Omega and
BLAST: `ebi_submit()` starts a job and returns its job id, and
`ebi_result()` waits for the job to finish and fetches its result.
EBI asks for an e-mail address with every job, so that they can contact
you if a job causes problems; you may put your own address in `EMAIL`.


In [1]:
# Install the libraries Colab does not come with (does nothing if they are already installed)
import importlib.util, subprocess, sys
for module, package in [("Bio", "biopython")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [2]:
import time
import requests

EBI = "https://www.ebi.ac.uk/Tools/services/rest"
EMAIL = "kb8029-student@example.org"   # EBI asks for an e-mail address with every job; you may use your own

def ebi_submit(tool, **params):
    """Start a job with one of EBI's tools (for example 'clustalo') and return its job id."""
    r = requests.post(f"{EBI}/{tool}/run", data={"email": EMAIL, **params}, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"EBI did not accept the {tool} job: {r.text[:300]}")
    return r.text

def ebi_result(tool, job, result_type, max_minutes=10):
    """Wait until the job has finished, then return one of its results as text."""
    start = time.time()
    while True:
        status = requests.get(f"{EBI}/{tool}/status/{job}", timeout=60).text
        if status == "FINISHED":
            break
        if status not in ("QUEUED", "RUNNING"):
            raise RuntimeError(f"{tool} job {job} ended with status {status}")
        minutes = (time.time() - start) / 60
        if minutes > max_minutes:
            raise TimeoutError(f"EBI is busy: job {job} is still {status} after {max_minutes} minutes. "
                               "Run this cell again later; the job keeps running on EBI's side.")
        print(f"\r{tool}: {status.lower()} ({minutes:.1f} min)", end="")
        time.sleep(5)
    print(f"\r{tool}: finished after {(time.time() - start) / 60:.1f} min      ")
    r = requests.get(f"{EBI}/{tool}/result/{job}/{result_type}", timeout=120)
    r.raise_for_status()
    return r.text

## 2. Clustal Omega: a multiple sequence alignment, and a PSSM from it

Five human globin-family proteins (HBB, HBA1, MB, NGB, CYGB, the same
set this course has used since Sessions 2 and 4) aligned with **Clustal
Omega**, the current version of the classic progressive aligner that the
Session 4 page describes (ClustalW is its predecessor). Once you have a
real alignment, building a PSSM from it is just counting: for each
column, how often does each amino acid appear? The job usually takes
10-30 seconds.


In [3]:
import io
from collections import Counter
from Bio import AlignIO

GLOBIN_SEQS = {
    "HBB":  "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH",
    "HBA1": "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR",
    "MB":   "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIPVKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG",
    "NGB":  "MERPEPELIRQSWRAVSRSPLEHGTVLFARLFALEPDLLPLFQYNCRQFSSPEDCLSSPEFLDHIRKVMLVIDAAVTNVEDLSSLEEYLASLGRKHRAVGVKLSSFSTVGESLLYMLEKCLGPAFTPATRAAWSQLYGAVVQAMSRGWDGE",
    "CYGB": "MEKVPGEMEIERRERSEELSEAERKAVQAMWARLYANCEDVGVAILVRFFVNFPSAKQYFSQFKHMEDPLEMERSPQLRKHACRVMGALNTVVENLHDPDKVSSVLALVGKAHALKHKVEPVYFKILSGVILEVVAEEFASDFPPETQRAWAKLRGLIYSHVTAAYKEVGWVQQVPNATTPPATLPSSGP",
}
fasta = "".join(f">{name}\n{seq}\n" for name, seq in GLOBIN_SEQS.items())

job = ebi_submit("clustalo", sequence=fasta, stype="protein", outfmt="clustal_num")
aln_text = ebi_result("clustalo", job, "aln-clustal_num")
aln = AlignIO.read(io.StringIO(aln_text), "clustal")

print(f"Clustal Omega alignment: {len(aln)} sequences, {aln.get_alignment_length()} columns")
for rec in aln:
    print(f"  {rec.id:6s} {str(rec.seq)[:60]}...")

# A PSSM starts by counting how often each amino acid appears in each
# alignment column (gaps don't count as a residue).
total_counts = Counter()
for col in range(aln.get_alignment_length()):
    total_counts.update(r for r in aln[:, col] if r != "-")

print("\nTotal count per amino acid, summed across all columns:")
for aa, n in total_counts.most_common(10):
    print(f"  {aa}: {n}")

clustalo: queued (0.0 min)

clustalo: finished after 0.1 min      


Clustal Omega alignment: 5 sequences, 191 columns
  HBB    ---------------MVHLTPEEKSAVTALWGKVNV--DEVGGEALGRLLVVYPWTQRFF...
  HBA1   ----------------MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYF...
  MB     ----------------MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKF...
  NGB    ------------------MERPEPELIRQSWRAVSRSPLEHGTVLFARLFALEPDLLPLF...
  CYGB   MEKVPGEMEIERRERSEELSEAERKAVQAMWARLYANCEDVGVAILVRFFVNFPSAKQYF...

Total count per amino acid, summed across all columns:
  L: 89
  A: 81
  V: 70
  E: 59
  K: 58
  G: 51
  S: 51
  P: 41
  H: 37
  F: 37


## 3. An iterated profile search: round 1 against round 3

PSI-BLAST's whole point is that later rounds search with a **profile**,
built from the previous round's hits, instead of a single fixed
substitution matrix (the "From one sequence to many" section of the
Session 5 page). Here we use **jackhmmer**, the same idea from the HMMER
package: round 1 is an ordinary single-sequence search, and every later
round builds a profile HMM from the hits *included* so far (E-value below
0.01 by default) and searches again with it.

The query is alpha-globin (HBA1 and HBA2 make the same protein, UniProt
`P69905`), and the database is every protein sequence in the PDB (about
170,000 chains), through EBI's HMMER API. Three rounds take about ten
seconds.


In [4]:
HMMER = "https://www.ebi.ac.uk/Tools/hmmer/api/v1"

def jackhmmer(sequence, database="pdb", rounds=3, max_wait=300):
    """Run jackhmmer on EBI's HMMER server; return {round: list of hits}."""
    r = requests.post(f"{HMMER}/search/jackhmmer", timeout=60,
                      json={"input": ">query\n" + sequence, "database": database, "iterations": rounds})
    r.raise_for_status()
    job, results, start = r.json()["id"], {}, time.time()
    while len(results) < rounds:                    # each round is a job of its own, linked by next_job_id
        if time.time() - start > max_wait:
            raise TimeoutError("The HMMER server is slow right now. Wait a minute and run the cell again.")
        info = requests.get(f"{HMMER}/search/{job}", timeout=60).json()
        if info.get("task") and info["task"]["status"] == "SUCCESS" and info["iteration"]:
            hits = requests.get(f"{HMMER}/result/{job}", params={"page_size": 2000}, timeout=120).json()["result"]["hits"]
            results[info["iteration"]] = hits
        if info.get("next_job_id"):
            job = info["next_job_id"]
        else:
            time.sleep(2)
    return results

r = requests.get("https://rest.uniprot.org/uniprotkb/P69905.fasta", timeout=30)
r.raise_for_status()
HBA = "".join(r.text.splitlines()[1:])
print(r.text)

rounds = jackhmmer(HBA)
for n, hits in sorted(rounds.items()):
    included = [h for h in hits if h["is_included"]]
    print(f"Round {n}: {len(included)} significant PDB chains (included in the profile)")

>sp|P69905|HBA_HUMAN Hemoglobin subunit alpha OS=Homo sapiens OX=9606 GN=HBA1 PE=1 SV=2
MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG
KKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTP
AVHASLDKFLASVSTVLTSKYR



Round 1: 413 significant PDB chains (included in the profile)
Round 2: 563 significant PDB chains (included in the profile)
Round 3: 573 significant PDB chains (included in the profile)


Which proteins does the profile find that the single sequence did not?
The cell below lists the descriptions of the PDB chains that are
significant in round 3 but were not in round 1, with the species where
the PDB gives one.


In [5]:
from collections import Counter

chain = lambda h: h["metadata"]["accession"]          # PDB entry and chain, for example 1c7c_A
round1 = {chain(h) for h in rounds[1] if h["is_included"]}
new = [h for h in rounds[3] if h["is_included"] and chain(h) not in round1]
print(f"{len(new)} chains are significant in round 3 but not in round 1. The commonest descriptions:\n")
names = Counter(h["metadata"].get("description", "?").capitalize() for h in new)
for name, n in names.most_common(15):
    species = Counter(h["metadata"].get("species") for h in new if h["metadata"].get("description", "?").capitalize() == name)
    top = ", ".join(s for s, _ in species.most_common(2) if s)
    print(f"  {n:3d}  {name[:45]:<45} {top}")

161 chains are significant in round 3 but not in round 1. The commonest descriptions:

   22  Neuroglobin                                   Mus musculus, Homo sapiens
   18  Globin-1                                      
   13  Dehaloperoxidase a                            Amphitrite ornata
    9  Myoglobin                                     Physeter macrocephalus, Aplysia limacina
    8  Neural hemoglobin                             Cerebratulus lacteus
    7  Hemoglobin                                    Gasterophilus intestinalis
    6  Dehaloperoxidase b                            Amphitrite ornata
    3  Globin i                                      
    3  Non-symbiotic hemoglobin 1                    Oryza sativa, Arabidopsis thaliana
    3  Non-symbiotic hemoglobin                      Trema tomentosum, Oryza sativa
    3  Hemoglobin ii                                 Phacoides pectinatus
    3  Hemoglobin iii                                Phacoides pectinatus
    3  Extracel

## 4. A live Pfam domain search: HIV-1 Pol

The same live EBI InterPro API that the Session 5 page's worked example
uses for *HBB*, here run on HIV-1's Gag-Pol polyprotein (UniProt
`P04585`). Unlike a single-domain protein such as HBB, it carries
several Pfam domains strung together: protease, reverse transcriptase,
RNase H and integrase, each a separately evolved enzymatic module.


In [6]:
r = requests.get("https://www.ebi.ac.uk/interpro/api/entry/pfam/protein/uniprot/P04585/", timeout=30)
r.raise_for_status()
results = r.json()["results"]

print(f"{len(results)} Pfam domains found for P04585 (HIV-1 Gag-Pol polyprotein):")
print()
hits = []
for entry in sorted(results, key=lambda e: e["proteins"][0]["entry_protein_locations"][0]["fragments"][0]["start"]):
    meta = entry["metadata"]
    loc = entry["proteins"][0]["entry_protein_locations"][0]   # InterPro calls the domain E-value "score"
    frag = loc["fragments"][0]
    hits.append((meta["accession"], meta["name"], frag["start"], frag["end"], loc["score"]))
    print(f"  {meta['accession']}  {meta['name']:38s} residues {frag['start']:>5}-{frag['end']:<5}  E-value={loc['score']}")

best = min(hits, key=lambda h: h[4])
print(f"\nMost significant (lowest E-value) match: {best[0]} \"{best[1]}\"")

11 Pfam domains found for P04585 (HIV-1 Gag-Pol polyprotein):

  PF00540  gag gene protein p17 (matrix protein)  residues     2-132    E-value=1.6e-61
  PF19317  Gag protein p24 C-terminal domain      residues   276-349    E-value=8e-29
  PF00098  Zinc knuckle                           residues   391-406    E-value=0.041
  PF00077  Retroviral aspartyl protease           residues   494-585    E-value=1.1e-23
  PF00078  Reverse transcriptase (RNA-dependent DNA polymerase) residues   650-821    E-value=8.2e-37
  PF06817  Reverse transcriptase thumb domain     residues   828-890    E-value=4.5e-19
  PF06815  Reverse transcriptase connection domain residues   905-1006   E-value=1.3e-41
  PF00075  RNase H                                residues  1025-1143   E-value=6.2e-28
  PF02022  Integrase Zinc binding domain          residues  1155-1191   E-value=4.9e-12
  PF00665  Integrase core domain                  residues  1204-1292   E-value=1.2e-10
  PF00552  Integrase DNA binding domain       

## Done

Once all three sections above have run and printed real output, answer
the Session 5 lab quiz on Canvas using your own results. Your numbers
may differ slightly from other students' if EBI has updated a database
or a program since the quiz was written; that is expected and fine.
